# Conversao ODBC para formato de fechamento

Converte qualquer export ODBC (CSV ou XLSX) para o layout padrao do fechamento geral.

## Como usar
1. Ajuste `BASE_PATH` para o arquivo ODBC de entrada (CSV ou XLSX).
2. Se for XLSX com varias abas, informe `BASE_SHEET` ou deixe `None` para usar a primeira aba.
3. Ajuste `MESES_REFERENCIA` (`MM/AAAA`). Use `None` para converter **tudo** sem filtrar por mes.
4. Ajuste `CAMPO_DATA_FILTRO` se necessario (`lancamento` ou `ite_pagrec_vencimento`).
5. Execute todas as celulas.

## Mapeamento de valores
- `valor_conta` <- `valor_centro`
- `valor_pago` <- `valor_bruto`
- `Valor Oficial` <- `valor_plano`

## Saida
- Por mes: `02-Referencias/Fechamento/FECHAMENTO_ODBC_{ano}_{mes}.xlsx`
- Consolidado (quando ha mais de um mes): `FECHAMENTO_ODBC_COMPLETO_{ano}.xlsx`
- Sem filtro de mes: `{stem_do_arquivo}_fechamento.xlsx`


In [15]:
NOTEBOOK_VERSAO = '2.0'  # sem arquivo MODELO externo

from pathlib import Path
import re
import pandas as pd
import numpy as np

# --- Parametros principais (ajuste aqui) ---
BASE_PATH = Path('..') / '..' / '02-Referencias' / 'CorporativoSucata_Apoio_ODBC.xlsx'
BASE_SHEET = 'Planilha1'          # None = primeira aba do XLSX
MESES_REFERENCIA = ['04/2026']  # None = sem filtro por mes
CAMPO_DATA_FILTRO = 'lancamento'       # 'lancamento' ou 'ite_pagrec_vencimento'
OUT_DIR = Path('..') / '..' / '02-Referencias'
PREFIXO_SAIDA = 'FECHAMENTO_ODBC'      # prefixo dos arquivos gerados por mes

# Layout padrao do fechamento (nao depende de arquivo modelo externo)
COLUNAS_FECHAMENTO = [
    'id', 'Segmento',
    'n1_cod_centro_custo', 'n1_centro_custo', 'n1_CC',
    'n2_cod_centro_custo', 'n2_centro_custo', 'n2_CC',
    'n3_cod_centro_custo', 'n3_centro_custo', 'n3_CC',
    'n4_cod_centro_custo', 'n4_centro_custo', 'n4_CC',
    'cod_conta', 'conta', 'cod_conta-descr',
    'filial', 'titulo', 'valor_nf', 'valor_pago', 'valor_conta',
    'observacao', 'data_nf', 'data_pagamento',
    'cod_credor_forn_cli_func', 'credor_forn_cli_func',
    'Origem', 'Sistema', 'Dados auxiliares', 'Valor Oficial',
    'DE-PARA1', 'DE-PARA2', 'CUSTEIO VARIÁVEL',
]

COLUNAS_ODBC_OBRIGATORIAS = [
    'codcen', 'descen', 'codcdc', 'descdc', 'filial', 'documento',
    'valor_bruto', 'valor_plano', 'valor_centro', 'observacao', 'lancamento',
    'iterea_pagamento', 'codigo_pessoa', 'nome', 'nota',
]

# Colunas do Excel de saida com tipo/formato fixo (letras = layout padrao do fechamento)
# T/U/V = valor_nf, valor_pago, valor_conta | X/Y = data_nf, data_pagamento | AE = Valor Oficial
COLUNAS_NUMERICAS_SAIDA = ('valor_nf', 'valor_pago', 'valor_conta', 'Valor Oficial')
COLUNAS_DATA_SAIDA = ('data_nf', 'data_pagamento')
FMT_NUMERICO_EXCEL = '#.##0,00'
FMT_DATA_BR_EXCEL = 'DD/MM/YYYY'

BASE_PATH = Path(BASE_PATH)
OUT_DIR = Path(OUT_DIR)

if not BASE_PATH.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {BASE_PATH.resolve()}')

print(f'Notebook v{NOTEBOOK_VERSAO} — sem arquivo MODELO externo')
print('Parametros carregados com sucesso.')
print(f'BASE={BASE_PATH.name} | MESES={MESES_REFERENCIA or "todos"} | CAMPO_DATA_FILTRO={CAMPO_DATA_FILTRO}')
print(f'Layout de saida: {len(COLUNAS_FECHAMENTO)} colunas (fixo no notebook)')


Notebook v2.0 — sem arquivo MODELO externo
Parametros carregados com sucesso.
BASE=CorporativoSucata_Apoio_ODBC.xlsx | MESES=['04/2026'] | CAMPO_DATA_FILTRO=lancamento
Layout de saida: 34 colunas (fixo no notebook)


In [16]:
def to_float_br(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    s = str(v).strip().replace('R$', '').replace(' ', '')
    if s == '':
        return np.nan
    # BR com milhar: 1.234,56 | Excel/CSV US: 174.08 | BR simples: 174,08
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

def to_br_number(v):
    if pd.isna(v):
        return ''
    s = f'{float(v):.2f}'
    return s.replace('.', ',')

def to_br_currency(v):
    if pd.isna(v):
        return ''
    txt = f'{float(v):,.2f}'
    txt = txt.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'R$ {txt}'

def split_descen(descen):
    partes = [p.strip() for p in str(descen).split('/') if p.strip()]
    natureza = partes[0] if len(partes) > 0 else ''
    divisao = partes[1] if len(partes) > 1 else ''
    filial_cc = partes[2] if len(partes) > 2 else ''
    resto = partes[3:] if len(partes) > 3 else []
    n3 = resto[0] if len(resto) > 0 else filial_cc
    n4 = ' / '.join(resto) if len(resto) > 0 else n3
    # ODBC as vezes repete o nome do nivel 3 no inicio de n4 (ex.: "PRENSA MOVEL / QXH2G14 (PHH0061)")
    if n3 and isinstance(n4, str) and n4.startswith(n3 + ' / '):
        n4 = n4[len(n3) + 3:].strip()
    # Quando n3 perde sufixo (ex.: JOINVILLE/SC), n4 pode ficar "SC / EHH0044"
    if isinstance(n4, str) and ' / ' in n4:
        tail = n4.split(' / ')[-1].strip()
        if re.match(
            r'^([A-Z]{3}\d{4}|[A-Z]{3}\d[A-Z0-9]{3}|[A-Z0-9]{5,8}\s*\([^)]+\))$',
            tail,
            re.IGNORECASE,
        ):
            n4 = tail
    if isinstance(n4, str) and ' / ' in n4 and 'operadores' in n4.lower() and 'arcelor' in n4.lower():
        n4 = 'ARCELOR RESENDE - OPERADORES'
    return natureza, divisao, filial_cc, n3, n4

def levels_from_codcen(codcen):
    tokens = [t for t in str(codcen).strip().split('.') if t]
    if len(tokens) < 2:
        n1 = str(codcen).strip()
    else:
        n1 = '.'.join(tokens[:2])
    n2 = '.'.join(tokens[:3]) if len(tokens) >= 3 else n1
    n3 = '.'.join(tokens[:4]) if len(tokens) >= 4 else n2
    n4 = '.'.join(tokens) if len(tokens) >= 1 else ''
    return n1, n2, n3, n4

def parse_data_nf(v):
    s = str(v).strip()
    if not s or set(s) == {'#'}:
        return pd.NaT
    dt = pd.to_datetime(s, format='%d/%m/%Y', errors='coerce')
    if pd.isna(dt):
        dt = pd.to_datetime(s, errors='coerce')
    return dt


def parse_data_pagamento(v):
    s = str(v).strip()
    if not s or s.upper() == 'NAO PAGO' or s in {'1800-01-01', '1900-01-01'}:
        return pd.NaT
    return pd.to_datetime(s, errors='coerce')


def fmt_data_nf(v):
    dt = parse_data_nf(v)
    if pd.isna(dt):
        return ''
    return dt.strftime('%d/%m/%Y')


def fmt_data_pagamento(v):
    dt = parse_data_pagamento(v)
    if pd.isna(dt):
        return ''
    return dt.strftime('%d/%m/%Y')


In [17]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import gravar_fechamento_excel


def salvar_fechamento(df_out: pd.DataFrame, caminho: Path) -> None:
    gravar_fechamento_excel(df_out, caminho, sheet_name='Fechamento')


Fonte ODBC: CorporativoSucata_Apoio_ODBC.xlsx (xlsx) | linhas: 358
Colunas ODBC: 18 | layout fechamento: 34 colunas
04/2026: ODBC valor_bruto=30,059.31 | saida valor_pago=30,059.31
04/2026: ODBC valor_plano=-20,758.74 | saida Valor Oficial=-20,758.74
04/2026: ODBC valor_centro=-20,758.74 | saida valor_conta=-20,758.74
04/2026: 78 registros -> FECHAMENTO_ODBC_2026_04.xlsx


,id,Segmento,n1_cod_centro_custo,n1_centro_custo,n1_CC,n2_cod_centro_custo,n2_centro_custo,n2_CC,n3_cod_centro_custo,n3_centro_custo,...,data_pagamento,cod_credor_forn_cli_func,credor_forn_cli_func,Origem,Sistema,Dados auxiliares,Valor Oficial,DE-PARA1,DE-PARA2,CUSTEIO VARIÁVEL
0,000001,SELETIVA,1.2,SELETIVA,1.2 SELETIVA,1.2.1,CORPORATIVO SUCATA,1.2.1 CORPORATIVO SUCATA,1.2.1.1,APOIO,...,01/04/2026,121,BANCO COOPERATIVO SICREDI SA,Saida (Aplicacoes),SAGI,,"R$ -21,49",,,
1,000002,SELETIVA,1.2,SELETIVA,1.2 SELETIVA,1.2.1,CORPORATIVO SUCATA,1.2.1 CORPORATIVO SUCATA,1.2.1.1,APOIO,...,01/04/2026,121,BANCO COOPERATIVO SICREDI SA,Saida (Aplicacoes),SAGI,,"R$ -7,21",,,
2,000003,SELETIVA,1.2,SELETIVA,1.2 SELETIVA,1.2.1,CORPORATIVO SUCATA,1.2.1 CORPORATIVO SUCATA,1.2.1.1,APOIO,...,06/04/2026,121,BANCO COOPERATIVO SICREDI SA,Saida (Aplicacoes),SAGI,,"R$ -9,00",,,
3,000004,SELETIVA,1.2,SELETIVA,1.2 SELETIVA,1.2.1,CORPORATIVO SUCATA,1.2.1 CORPORATIVO SUCATA,1.2.1.1,APOIO,...,09/04/2026,121,BANCO COOPERATIVO SICREDI SA,Saida (Aplicacoes),SAGI,,"R$ -18,51",,,
4,000005,SELETIVA,1.2,SELETIVA,1.2 SELETIVA,1.2.1,CORPORATIVO SUCATA,1.2.1 CORPORATIVO SUCATA,1.2.1.1,APOIO,...,09/04/2026,121,BANCO COOPERATIVO SICREDI SA,Saida (Aplicacoes),SAGI,,"R$ -8,19",,,


In [18]:
# Checagem rapida de aderencia ao layout
faltantes = [c for c in COLUNAS_FECHAMENTO if c not in out.columns]
extras = [c for c in out.columns if c not in COLUNAS_FECHAMENTO]
print('Colunas faltantes:', faltantes)
print('Colunas extras:', extras)
print('Quantidade de colunas esperadas:', len(COLUNAS_FECHAMENTO))
print('Quantidade de colunas na saida:', len(out.columns))

print()
if arquivos_gerados:
    print('Arquivo(s) gerado(s):')
    for caminho in arquivos_gerados:
        print(f'  - {caminho.resolve()}')
else:
    print('Nenhum arquivo gerado (execute a celula anterior).')


Colunas faltantes: []
Colunas extras: []
Quantidade de colunas esperadas: 34
Quantidade de colunas na saida: 34

Arquivo(s) gerado(s):
  - C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\FECHAMENTO_ODBC_2026_04.xlsx
